### Phase 4: Counterfactual Analysis

In this phase, counterfactual analysis is performed to understand how changes in the treatment variable affect the outcome at an individual level.

Instead of relying only on the estimated causal effect, this phase simulates "what-if" scenarios by modifying the treatment value and observing the resulting change in the predicted outcome.

In [1]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestRegressor

C:\Users\vinee\.conda\envs\ds\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
student_df = pd.read_csv("../data/raw/student-mat.csv", sep = ";")
print("Data loaded:", student_df.shape)

Data loaded: (395, 33)


In [3]:
with open("../artifacts/phase1_data_understanding/data_schema.pkl", "rb") as f:
    SCHEMA = pickle.load(f)
with open("../artifacts/phase2_dag_generation/causal_dag.pkl", "rb") as f:
    causal_graph = pickle.load(f)
with open("../artifacts/phase3_causal_estimation/causal_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("../artifacts/phase3_causal_estimation/causal_effect.pkl", "rb") as f:
    estimate = pickle.load(f)
print("All artifacts loaded successfully")

All artifacts loaded successfully


C:\Users\vinee\.conda\envs\ds\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading Required Artifacts

The artifacts generated from previous phases are loaded, including the data schema, causal graph, causal model, and estimated effect.

These components ensure that the counterfactual analysis is consistent with the previously defined causal structure.

In [4]:
sample = student_df.iloc[0:1].copy()
print("Original Student:")
sample

Original Student:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6


### Selecting Sample Instance

A single data instance is selected from the dataset to perform counterfactual analysis.

This instance represents the original state before applying any intervention.

In [5]:
cf_sample = sample.copy()
cf_sample["studytime"] += 1
print("Counterfactual Student:")
cf_sample

Counterfactual Student:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6


### Creating Counterfactual Scenario

A counterfactual version of the selected sample is created by modifying the treatment variable.

This represents a hypothetical scenario where the treatment value is changed while keeping all other features constant.

In [6]:
X = student_df[SCHEMA["features"]]
y = student_df[SCHEMA["outcome"]]
reg = RandomForestRegressor(random_state = 42)
reg.fit(X, y)
print("Model trained")

Model trained


### Training Predictive Model

A Random Forest Regressor is trained on the dataset to learn the relationship between input features and the outcome variable.

This model is used to generate predictions for both original and counterfactual scenarios.

In [7]:
original_pred = reg.predict(sample[SCHEMA["features"]])[0]
cf_pred = reg.predict(cf_sample[SCHEMA["features"]])[0]
print("Original Prediction:", original_pred)
print("Counterfactual Prediction:", cf_pred)
print("Effect:", cf_pred - original_pred)

Original Prediction: 7.977749999999999
Counterfactual Prediction: 11.815
Effect: 3.837250000000001


### Generating Predictions

Predictions are generated for both the original sample and the counterfactual sample using the trained model.

These predictions represent the expected outcome under actual and modified conditions.

### Individual Counterfactual Effect

The difference between the counterfactual prediction and the original prediction represents the individual-level causal effect.

This quantifies how the outcome changes when the treatment is modified for a specific instance.

In [8]:
effects = []
for i in range(len(student_df)):
    sample = student_df.iloc[i:i+1].copy()
    cf_sample = sample.copy()
    cf_sample["studytime"] += 1
    original = reg.predict(sample[SCHEMA["features"]])[0]
    cf = reg.predict(cf_sample[SCHEMA["features"]])[0]
    effects.append(cf - original)
print("Average Individual Effect:", sum(effects) / len(effects))

Average Individual Effect: 0.6018969487487349


### Individual Effects Across Dataset

The counterfactual analysis is extended to all instances in the dataset by iterating through each sample.

This produces a set of individual effects, capturing variation across different data points.

### Average Counterfactual Effect

The average of all individual effects is computed to obtain an overall estimate of the treatment impact based on the counterfactual model.

This value can be compared with the causal effect estimated in the previous phase.

In [9]:
with open("../artifacts/phase4_counterfactual_engine/counterfactual_model.pkl", "wb") as f:
    pickle.dump(reg, f)
with open("../artifacts/phase4_counterfactual_engine/individual_effects.pkl", "wb") as f:
    pickle.dump(effects, f)
with open("../artifacts/phase4_counterfactual_engine/average_effect.pkl", "wb") as f:
    pickle.dump(sum(effects) / len(effects), f)

### Saving Results

The trained counterfactual model, individual effects, and average effect are saved for use in later phases.

This allows reuse without recomputing the entire process.

### Conclusion

This phase provides a data-driven view of causal impact by simulating counterfactual scenarios.

It complements the causal estimation phase by offering insight into how individual outcomes change under different treatment conditions.